# 01. BIRD 데이터 추출

BIRD dev 데이터셋의 SQLite DB 를 테이블별 CSV 로 풀고, 공식 `dev_tables.json` 의 FK 정의로
DB 별 정답(`ground_truth.json`)을 만든다. `db_name` 을 평가 대상 DB 마다 바꿔 실행한다.

- 평가 DB: `toxicology`, `california_schools`, `superhero`, `thrombosis_prediction`
- 입력: `data/bird/dev_databases/<db>/<db>.sqlite`, `data/bird/dev_tables.json`
- 출력: `workspace/dataset_mini/bird_data_tables/<db>/*.csv`, `ground_truth.json`

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

def extract_sqlite_tables_to_csv(db_path, output_dir):
    """SQLite 데이터베이스의 모든 테이블을 CSV로 저장"""
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    
    # 모든 테이블 조회
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    db_name = Path(db_path).stem
    db_output_dir = f"{output_dir}/{db_name}"
    Path(db_output_dir).mkdir(exist_ok=True)
    
    print(f"\n✅ {db_name} - {len(tables)}개 테이블")
    
    for table_name in tables:
        table_name = table_name[0]
        # if table_name == "order":
        #     pass
        # else:
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        csv_file = f"{db_output_dir}/{table_name}.csv"
        df.to_csv(csv_file, index=False, encoding='utf-8')
        print(f"   • {table_name}: {len(df):,}행")
    
    conn.close()

# db_name = "superhero"
# db_name = "financial"
# db_name = "codebase_community"
db_name = "toxicology"  # toxicology / california_schools / superhero / thrombosis_prediction

# 사용 예시
extract_sqlite_tables_to_csv(
    # "../data/bird/dev_databases/california_schools/california_schools.sqlite",
    f"../data/bird/dev_databases/{db_name}/{db_name}.sqlite",
    "./dataset_mini/bird_data_tables"
)

# extract_sqlite_tables_to_csv(
#     "mini_dev_data/dev_databases/trip_data/trip_data.sqlite",
#     "./bird_data_tables"
# )

In [ ]:
import json

# db_name = "financial"
db_name = "toxicology"  # toxicology / california_schools / superhero / thrombosis_prediction
# db_name = "thrombosis_prediction"

def extract_bird_fk_ground_truth(tables_json_path, target_db=db_name):
    # BIRD 공식 tables.json 로드
    with open(tables_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    for db in data:
        if db['db_id'] == target_db:
            table_names = db['table_names_original']
            column_names = db['column_names_original'] # [[table_idx, "col_name"], ...]
            foreign_keys = db['foreign_keys']          # [[col_idx1, col_idx2], ...]
            
            gt_json = []
            for fk in foreign_keys:
                idx1, idx2 = fk[0], fk[1]
                
                # 인덱스를 기반으로 실제 테이블명과 컬럼명 매핑
                t1_idx, c1 = column_names[idx1]
                t2_idx, c2 = column_names[idx2]
                
                node1 = f"{table_names[t1_idx]}.{c1}"
                node2 = f"{table_names[t2_idx]}.{c2}"
                
                gt_json.append([node1, node2])
                
            return gt_json
    
    print(f"Database '{target_db}' not found in tables.json")
    return []

# 실행 및 저장"
bird_gt_json = extract_bird_fk_ground_truth('../data/bird/dev_tables.json', db_name)

# 결과 출력
print(json.dumps(bird_gt_json, indent=2))

# 파일로 저장 (평가용)
with open(f"./dataset_mini/bird_data_tables/{db_name}/ground_truth.json", 'w') as f:
    json.dump(bird_gt_json, f, indent=2)